In [ ]:
# ============================================================
# CELL 1: Setup & Load Model (run once)
# ============================================================
import os
import time
import torch
import numpy as np
import cv2
from gym_hil.wrappers.intervention_utils import GamepadController
from lerobot.configs import PreTrainedConfig
from lerobot.policies import make_policy, make_pre_post_processors
from lerobot.envs.utils import preprocess_observation
from lerobot.datasets import LeRobotDatasetMetadata
from lerobot.utils.constants import ACTION
from organizing_env import DeskOrganizerEnv
from lerobot.processor.pipeline import ProcessorStepRegistry



os.environ['MUJOCO_GL'] = 'egl'

_original_update = GamepadController.update
def _safe_update(self):
    if self.controller_config is None:
        return
    _original_update(self)
GamepadController.update = _safe_update

_original_should_intervene = GamepadController.should_intervene
def _safe_should_intervene(self):
    if self.controller_config is None:
        return True
    return _original_should_intervene(self)
GamepadController.should_intervene = _safe_should_intervene

# Config
CHECKPOINT = "KeshavLN/deskorgv1.4_policy"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

policy_cfg = PreTrainedConfig.from_pretrained(CHECKPOINT)
policy_cfg.device = DEVICE
policy_cfg.pretrained_path = CHECKPOINT
policy_cfg.n_action_steps = 10


ds_meta = LeRobotDatasetMetadata("KeshavLN/deskorgv1.2_dataset")
policy = make_policy(cfg=policy_cfg, ds_meta=ds_meta)
policy.eval()
print("Policy loaded.")

try:
    ProcessorStepRegistry.get("delta_actions_processor")
except KeyError:
    RelativeActionsStep = ProcessorStepRegistry.get("relative_actions_processor")
    ProcessorStepRegistry._registry["delta_actions_processor"] = RelativeActionsStep

preprocessor, postprocessor = make_pre_post_processors(
    policy_cfg=policy_cfg,
    pretrained_path=CHECKPOINT,
    preprocessor_overrides={"device_processor": {"device": DEVICE}},
)
print("Processors loaded. Ready for rollouts!")

In [ ]:
# ============================================================
# CELL 2: Run Rollout (re-run this cell for each new episode)
# ============================================================

TASK = "Pick up the yellow cube and place it on the red circle."
TARGET_OBJECT = "yellow_cube"
MAX_STEPS = 500
SAVE_VIDEO = "rollout.mp4"    # set to None to skip saving

env = DeskOrganizerEnv(
    task_desc=TASK,
    render_mode="rgb_array",
    image_obs=True,
    use_gripper=True,
    control_mode="manual",
    target_object=TARGET_OBJECT,
)

video_writer = None
video_writer_eih = None
if SAVE_VIDEO:
    fps = env._env.control_freq
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video_writer = {'fourcc': fourcc, 'fps': fps, 'path': SAVE_VIDEO, 'writer': None}
    base, ext = os.path.splitext(SAVE_VIDEO)
    eih_path = f'{base}_eye_in_hand{ext}'
    video_writer_eih = {'fourcc': fourcc, 'fps': fps, 'path': eih_path, 'writer': None}

policy.reset()
obs, info = env.reset()
step = 0
done = False

try:
    while not done and step < MAX_STEPS:
        start_time = time.time()

        if SAVE_VIDEO:
            for vw, img_key in [(video_writer, 'image'), (video_writer_eih, 'image2')]:
                frame = obs['pixels'][img_key]
                if vw['writer'] is None:
                    h, w = frame.shape[:2]
                    vw['writer'] = cv2.VideoWriter(vw['path'], vw['fourcc'], vw['fps'], (w, h))
                vw['writer'].write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))

        # Preprocess → predict → postprocess
        obs_torch = preprocess_observation(obs)

        obs_torch['task'] = [env.task]

        # Fix for Libero checkpoints: If the environment outputs 9 dims but the model 
        # expects 8 dims, slice off the duplicate second gripper joint.
        expected_dim = policy.config.input_features["observation.state"].shape[0]
        if obs_torch["observation.state"].shape[-1] == 9 and expected_dim == 8:
            obs_torch["observation.state"] = obs_torch["observation.state"][..., :8]
                

        obs_torch = preprocessor(obs_torch)

        with torch.no_grad():
            action = policy.select_action(obs_torch)

        action = postprocessor(action)

        if isinstance(action, dict):
            action_tensor = action[ACTION]
        else:
            action_tensor = action
        action_numpy = action_tensor.squeeze(0).to('cpu').numpy()

        obs, reward, terminated, truncated, info = env.step(action_numpy)
        done = terminated or truncated

        step += 1
        if step % 20 == 0:
            print(f'Step {step}/{MAX_STEPS} | Success: {info.get("is_success", False)}')

        if info.get('is_success', False):
            print(f'\n[SUCCESS] Task accomplished in {step} steps!')
            break

        elapsed = time.time() - start_time
        time.sleep(max(0, (1.0 / env._env.control_freq) - elapsed))

except KeyboardInterrupt:
    print('\nRollout interrupted.')
finally:
    if video_writer and video_writer['writer']:
        video_writer['writer'].release()
        print(f"Frontview video saved to: {video_writer['path']}")
    if video_writer_eih and video_writer_eih['writer']:
        video_writer_eih['writer'].release()
        print(f"Eye-in-hand video saved to: {video_writer_eih['path']}")
    env.close()
    print(f'Episode finished at step {step}.')